# Gold: hospital_daily_summary

Builds `project1_gs_caresync.gold.hospital_daily_summary` (one row per hospital per day) per the STTM `sttm_silver_to_gold.xlsx`. Driving table: `silver.lab_results` grouped by hospital_id and test_date; LEFT JOINs to `hospitals` and `calendar`; LAG() for day-over-day trend. Full rebuild (CREATE OR REPLACE).

Schema notes: the calendar key column is `datekey` (INT, yyyyMMdd), not `date_key` as written in the STTM.

In [0]:
%sql
CREATE OR REPLACE TABLE project1_gs_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volumes, rates and day-over-day trend (STTM silver->gold)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM project1_gs_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(d.hospital_id AS INT)                                   AS hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)                                   AS total_tests,
  CAST(d.unique_patients_tested AS INT)                        AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                           AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                           AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / d.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / d.total_tests AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                          AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE (d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests
    END AS DECIMAL(6,2))                                       AS test_volume_change_pct
FROM with_lag d
LEFT JOIN project1_gs_caresync.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN project1_gs_caresync.silver.calendar c
  ON c.datekey = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT)

In [0]:
%sql
-- Grain check: (hospital_id, summary_date) unique; total tests reconcile to silver.lab_results
SELECT
  COUNT(*)                                          AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)         AS distinct_hospital_days,
  SUM(total_tests)                                  AS sum_total_tests,
  (SELECT COUNT(*) FROM project1_gs_caresync.silver.lab_results) AS silver_lab_rows,
  COUNT_IF(day_name IS NULL)                        AS missing_calendar_rows
FROM project1_gs_caresync.gold.hospital_daily_summary

In [0]:
updated_df = spark.table("project1_gs_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))